# 03 — Test risk-engine (pub/sub channel `risk_update`)

Smoke test du container `fxvol-risk` — étape 3/5. Valide que **l'engine publie sur le channel Redis `risk_update`** avec le format JSON attendu, à la cadence du cycle (2s par message, donc ~2-3 messages reçus sur une fenêtre d'écoute de 5s).

**Pourquoi c'est critique** : le channel `risk_update` est ce que `api/ws/redis_bridge.py` consume pour pousser les Greeks au frontend via WebSocket `/ws/risk`. Le notebook 02 a validé le **cache** Redis (`SET latest_greeks:portfolio`), ce notebook valide le **stream** (`PUBLISH risk_update`). Sans ce stream, le hook `useRiskStream` du dashboard ne reçoit rien en live.

**Couvre** :
1. Seed surface (réplique notebook 02 — sinon cycle skip et aucun PUBLISH)
2. Subscribe `risk_update` + listen 5s → recevoir ≥ 2 messages (cycle 2s = 2-3 max théorique)
3. Tous les messages JSON-parseables
4. Schéma identique au cache : `{timestamp, greeks:{delta, gamma, vega, theta, spot}}`
5. Types corrects (timestamp str, greeks values floats finis)
6. Cadence ~2s : intervalle médian entre 2 messages consécutifs ∈ [1.5s, 3s]

**Préreq** :
- Notebook 02 vert
- `latest_spot:EURUSD` présent en Redis (market-data en cours)

## Setup + seed surface

In [1]:
import json
import math
import time
from datetime import datetime, timezone
from statistics import median

import redis

REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"
CHANNEL = "risk_update"
EXPECTED_GREEKS_KEYS = {"delta", "gamma", "vega", "theta", "spot"}

# Cycle risk-engine = 2s. Sur 5s on attend 2-3 messages.
MIN_MESSAGES = 2
LISTEN_DURATION_S = 5.0

# Cadence cycle = 2s ± latence Python/réseau. Fenêtre [1.5, 3] s = marge généreuse.
MIN_INTERVAL_MS = 1500
MAX_INTERVAL_MS = 3000

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps'")

# Seed inline pour que les cycles risk-engine ne skippent pas pendant l'écoute.
fake_surface = {
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "surface": {
        "1M": {"atm": {"iv": 0.07, "strike": 1.17}},
        "3M": {"atm": {"iv": 0.078, "strike": 1.17}},
    },
}
r.set(f"latest_vol_surface:{SYMBOL}", json.dumps(fake_surface), ex=600)
print(f"Connected -> {REDIS_URL}, channel = {CHANNEL!r}, surface seedé")

Connected -> redis://localhost:6380/0, channel = 'risk_update', surface seedé


## 1. Subscribe + listen 5s

**Ce que tu dois voir** : ≥ 2 messages reçus en 5s. Avec un cycle de 2s, on attend 2-3 messages selon le timing du subscribe vs le prochain cycle.

**Si 0 message** : engine ne publie pas. Soit cycle skip silencieusement (vérifier `latest_spot` ET surface présents en Redis), soit le bridge channel est pas bien configuré (vérifier `bus/channels.py:CH_RISK_UPDATE`).

In [2]:
print(f"== 1. subscribe + listen {LISTEN_DURATION_S}s ==")

sub = redis.from_url(REDIS_URL, decode_responses=True).pubsub(ignore_subscribe_messages=True)
sub.subscribe(CHANNEL)

messages = []  # liste de (timestamp_recv_ms, raw_data)
deadline = time.perf_counter() + LISTEN_DURATION_S
while time.perf_counter() < deadline:
    msg = sub.get_message(timeout=0.1)
    if msg and msg.get("type") == "message":
        messages.append((time.perf_counter() * 1000, msg["data"]))

sub.unsubscribe(CHANNEL)
sub.close()

record(f"≥ {MIN_MESSAGES} messages reçus en {LISTEN_DURATION_S}s",
       len(messages) >= MIN_MESSAGES,
       f"{len(messages)} messages")

== 1. subscribe + listen 5.0s ==
  [OK  ] ≥ 2 messages reçus en 5.0s  | 3 messages


## 2. JSON valide + schéma `{timestamp, greeks}`

Identique au notebook 02 §3 mais sur le payload reçu via PUBLISH au lieu du cache GET. Format attendu (cf. `bus/publisher.py:166`) :
```json
{"timestamp": "ISO", "greeks": {"delta":0.0, "gamma":0.0, "vega":0.0, "theta":0.0, "spot":1.17}}
```

In [3]:
print("== 2. JSON parsing + schema ==")

if not messages:
    record("JSON valide + clés attendues", False, "aucun message à inspecter (cf. §1)")
else:
    parse_errors = 0
    schema_errors = 0
    parsed_msgs = []
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            parse_errors += 1
            continue
        if {"timestamp", "greeks"} <= set(obj.keys()) and \
           EXPECTED_GREEKS_KEYS <= set(obj.get("greeks", {}).keys()):
            parsed_msgs.append(obj)
        else:
            schema_errors += 1
    record("tous les messages JSON-parseables",
           parse_errors == 0,
           f"{parse_errors} erreurs de parse / {len(messages)}")
    record("top-level {timestamp, greeks} + 5 keys greeks",
           schema_errors == 0,
           f"{schema_errors} schémas incomplets / {len(messages)}")
    if parsed_msgs:
        sample = parsed_msgs[0]
        print(f"  [INFO] sample[0] = {sample}")

== 2. JSON parsing + schema ==
  [OK  ] tous les messages JSON-parseables  | 0 erreurs de parse / 3
  [OK  ] top-level {timestamp, greeks} + 5 keys greeks  | 0 schémas incomplets / 3
  [INFO] sample[0] = {'timestamp': '2026-04-28T15:23:13.681607Z', 'greeks': {'delta': 0.0, 'gamma': 0.0, 'vega': 0.0, 'theta': 0.0, 'spot': 1.17083}}


## 3. Types corrects sur tous les messages

`timestamp` str ISO-parsable, `greeks.delta/gamma/vega/theta/spot` floats finis.

In [4]:
print("== 3. types ==")

if not messages:
    record("types", False, "skip (cf. §1)")
else:
    ts_errors = 0
    greek_errors = 0
    for _, raw in messages:
        try:
            obj = json.loads(raw)
        except json.JSONDecodeError:
            continue
        ts = obj.get("timestamp")
        if not isinstance(ts, str):
            ts_errors += 1
            continue
        try:
            datetime.fromisoformat(ts.replace("Z", "+00:00"))
        except ValueError:
            ts_errors += 1
        g = obj.get("greeks", {})
        for key in EXPECTED_GREEKS_KEYS:
            v = g.get(key)
            if not isinstance(v, (int, float)) or (isinstance(v, float) and math.isnan(v)):
                greek_errors += 1
                break
    record("timestamp ISO-8601 parsable",
           ts_errors == 0,
           f"{ts_errors} erreurs / {len(messages)}")
    record("greeks values floats finis",
           greek_errors == 0,
           f"{greek_errors} erreurs / {len(messages)}")

== 3. types ==
  [OK  ] timestamp ISO-8601 parsable  | 0 erreurs / 3
  [OK  ] greeks values floats finis  | 0 erreurs / 3


## 4. Cadence ~2s — intervalle médian dans la fenêtre attendue

Cycle risk-engine = `CYCLE_SECONDS = 2.0` (cf. `services/risk/engine.py`). Donc 2 messages consécutifs sur `risk_update` doivent être espacés de ~2s.

**Fenêtre [1.5s, 3s]** :
- 1.5s = marge inférieure (latence Python/scheduling/réseau)
- 3s = marge supérieure (cycle qui dérive un peu si calculs lourds, ou skip d'un cycle suite à un fetch de spot un peu lent)

**Métrique** : médiane des intervalles, robuste aux outliers.

In [5]:
print("== 4. cadence ~2s (médiane des intervalles) ==")

if len(messages) < 2:
    record("cadence", False, "skip (besoin de ≥ 2 messages)")
else:
    timestamps_ms = [t for t, _ in messages]
    intervals_ms = [t2 - t1 for t1, t2 in zip(timestamps_ms, timestamps_ms[1:])]
    med = median(intervals_ms)
    record(f"intervalle médian ∈ [{MIN_INTERVAL_MS}, {MAX_INTERVAL_MS}] ms",
           MIN_INTERVAL_MS <= med <= MAX_INTERVAL_MS,
           f"median = {med:.0f}ms ({len(intervals_ms)} intervals, min={min(intervals_ms):.0f}, max={max(intervals_ms):.0f})")

== 4. cadence ~2s (médiane des intervalles) ==
  [OK  ] intervalle médian ∈ [1500, 3000] ms  | median = 2001ms (2 intervals, min=1999, max=2002)


## Récap final

In [6]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<55} STATUS  DETAIL")
print("-" * 110)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<55} {sym:<6}  {detail}")
print("-" * 110)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — pub/sub stream sain. Pass au notebook 04 (WS bridge api).")


LABEL                                                   STATUS  DETAIL
--------------------------------------------------------------------------------------------------------------
≥ 2 messages reçus en 5.0s                              OK      3 messages
tous les messages JSON-parseables                       OK      0 erreurs de parse / 3
top-level {timestamp, greeks} + 5 keys greeks           OK      0 schémas incomplets / 3
timestamp ISO-8601 parsable                             OK      0 erreurs / 3
greeks values floats finis                              OK      0 erreurs / 3
intervalle médian ∈ [1500, 3000] ms                     OK      median = 2001ms (2 intervals, min=1999, max=2002)
--------------------------------------------------------------------------------------------------------------

6 OK / 0 FAIL  (6 total)

OK — pub/sub stream sain. Pass au notebook 04 (WS bridge api).


## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| 0 message en 5s | engine skip cycle silencieusement (spot ou surface manquant) | `redis-cli GET latest_spot:EURUSD` + `redis-cli GET latest_vol_surface:EURUSD` ; si OK, `docker logs fxvol-risk --tail 10` |
| `parse_errors > 0` | payload non-JSON ou serializer cassé | `redis-cli SUBSCRIBE risk_update` manuel pour voir le payload brut |
| schéma incomplet | `publish_risk_update` modifié sans aligner les consumers | revue `src/bus/publisher.py:158` |
| médiane < 1500ms | cycle plus rapide que prévu (CYCLE_SECONDS modifié) | check `src/engines/risk/engine.py` constante `CYCLE_SECONDS` |
| médiane > 3000ms | cycle dérive (calculs trop lourds ou Redis lent) | profiler `_aggregate_greeks` et `publish_risk_update` |